In [ ]:
!pip install graphviz -q

In [ ]:
import re
import graphviz
from IPython.display import display, clear_output
import ipywidgets as widgets

# 1. Flexible Node Data Structure
class DocNode:
    def __init__(self, name, node_type, content=""):
        self.name = name
        self.node_type = node_type  # "root", "internal", or "leaf"
        self.content = content
        self.children = []

# 2. Vector-less Markdown Structure Parser
def parse_markdown_to_tree(text):
    root = DocNode("DOCUMENT\n(Root)", "root")

    # Track the last active node at each heading level (up to 3 deep)
    # level 0 is root, level 1 is #, level 2 is ##, etc.
    active_nodes = {0: root}

    lines = [line.strip() for line in text.split('\n') if line.strip()]

    for line in lines:
        # Check for Markdown headers: # (H1), ## (H2), ### (H3)
        header_match = re.match(re.compile(r'^(#{1,3})\s+(.*)'), line)

        if header_match:
            level = len(header_match.group(1)) # Number of '#' determines hierarchy level
            title = header_match.group(2)

            new_node = DocNode(title, "internal")

            # Find the correct parent (the nearest higher-level node currently active)
            parent_level = level - 1
            while parent_level not in active_nodes and parent_level > 0:
                parent_level -= 1

            parent_node = active_nodes[parent_level]
            parent_node.children.append(new_node)

            # Update the active node tracker for this level
            active_nodes[level] = new_node

            # Clear out deeper active levels since we've broken into a new section
            for l in list(active_nodes.keys()):
                if l > level:
                    del active_nodes[l]
        else:
            # It's plain text content (Leaf Node)
            # Find the deepest active header node to attach this text to
            deepest_level = max(active_nodes.keys())
            parent_node = active_nodes[deepest_level]

            wrapped_text = line if len(line) < 40 else line[:37] + "..."
            leaf = DocNode(wrapped_text, "leaf", content=line)
            parent_node.children.append(leaf)

    return root

# 3. Responsive Renderer
def generate_graphviz_tree(root_node):
    dot = graphviz.Digraph(comment='Vectorless RAG Tree', graph_attr={'rankdir': 'TB'})
    dot.attr(nodesep='0.25', ranksep='0.4')
    dot.attr('node', fontname='Arial Bold', fontsize='11', shape='egg', style='filled', margin='0.2,0.1')

    node_counter = 0

    def add_to_graph(parent_id, current_node):
        nonlocal node_counter
        node_counter += 1
        current_id = f"node_{node_counter}"

        if current_node.node_type in ["root", "internal"]:
            dot.node(current_id, current_node.name, fillcolor='#D3D3D3', color='#888888', fontcolor='#111111')
        else:
            dot.node(current_id, current_node.name, fillcolor='#C8E6C9', color='#4CAF50', fontcolor='#1B5E20')

        if parent_id:
            dot.edge(parent_id, current_id, color='#555555', arrowsize='0.6')

        for child in current_node.children:
            add_to_graph(current_id, child)

    add_to_graph(None, root_node)
    return dot

# 4. Colab UI Widget Execution
upload_btn = widgets.FileUpload(accept='.txt', multiple=False, description="Upload Document")
output_area = widgets.Output()

def on_file_upload(change):
    with output_area:
        clear_output()
        uploaded_filename = list(upload_btn.value.keys())[0]
        raw_text = upload_btn.value[uploaded_filename]['content'].decode('utf-8')

        tree_root = parse_markdown_to_tree(raw_text)
        graph = generate_graphviz_tree(tree_root)

        print(f"Vector-less RAG parsed tree for: {uploaded_filename}\n")
        display(graph)

upload_btn.observe(on_file_upload, names='value')
display(upload_btn, output_area)

FileUpload(value={}, accept='.txt', description='Upload Document')

Output()